[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb)

# 🟠 Medium: Adam Optimizer

Implement the **Adam** optimizer from scratch.

### Signature
```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def step(self): ...
    def zero_grad(self): ...
```

### Algorithm (per parameter)
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [26]:
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = params
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.lr = lr
        self.beta1 = betas[0]
        self.beta2 = betas[1]
        self.t = [0] * len(self.params) #time-step per param
        self.eps = eps

    def step(self):
        with torch.no_grad():

          for i, p in enumerate(self.params):
            if p.grad is None:
              continue

            self.t[i] += 1
            grad = p.grad

            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * grad.square()


            m_hat = self.m[i] / (1 - self.beta1 ** self.t[i])
            v_hat = self.v[i] / (1 - self.beta2 ** self.t[i])

            p.add_(- self.lr * (m_hat / (torch.sqrt(v_hat) + self.eps))) #add in place for tensors

    def zero_grad(self):
        for p in self.params:
          if p.grad is not None:
            p.grad.zero_()

In [27]:
# 🧪 Debug
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'Step {i}: loss={loss.item():.4f}')

Step 0: loss=12.5974
Step 1: loss=12.3944
Step 2: loss=12.1939
Step 3: loss=11.9960
Step 4: loss=11.8005


In [28]:
# ✅ SUBMIT
from torch_judge import check
check('adam')


🧪 Testing: Adam Optimizer (Medium)
──────────────────────────────────────────────────
  ✅ [1/3] Parameters change after step (5.9ms)
  ✅ [2/3] Matches torch.optim.Adam (5.9ms)
  ✅ [3/3] zero_grad works (0.6ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (12.4ms total)
  Progress saved. Run status() to see your dashboard.

